Contexte 
Une entreprise exploite plusieurs bâtiments intelligents équipés de capteurs IoT. Chaque capteur 
collecte régulièrement des informations sur la température, l'humidité, la qualité de l'air, la 
consommation énergétique, le nombre de personnes présentes, le type de bâtiment, le mode de 
fonctionnement et l'état du système de climatisation. 
Les données collectées sont destinées à alimenter ultérieurement un modèle de Machine Learning 
capable de prédire la consommation énergétique ou de détecter les situations anormales. 
Cependant, les données brutes présentent volontairement différents problèmes : valeurs 
manquantes, doublons, valeurs aberrantes, types incorrects, valeurs incohérentes, variables 
catégorielles, catégories rares, déséquilibre des classes et échelles différentes entre variables. 
L'objectif de l'atelier est donc de transformer le fichier brut en un jeu de données propre et prêt 
pour le Machine Learning.

Partie 1 – Explorer les données 

In [10]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

1) Charger les données CSV 


In [11]:
df=pd.read_csv('../data/smart_building_raw.csv')

2) Afficher les premières lignes du dataset 

In [12]:
df.head()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
0,1174,2025-02-13 06:00:00,B8,Entrepôt,A,23.0,40.0,851.0,26.0,138.4,Eco,Normal,Jeudi,Non
1,1275,2025-03-10 12:00:00,B7,Bureau,D,28.0,69.0,538.0,0.0,170.2,Eco,Normal,Lundi,Non
2,1493,2025-05-04 00:00:00,B5,Centre commercial,B,19.5,52.3,1198.0,64.0,149.8,Normal,Normal,Dimanche,Oui
3,1073,2025-01-19 00:00:00,B6,Université,D,20.0,58.4,1014.0,30.0,141.6,Normal,Normal,Dimanche,Non
4,1454,2025-04-24 06:00:00,B7,Bureau,D,26.8,70.5,628.0,42.0,228.3,Normal,Normal,Jeudi,Non


3) Afficher les dernières lignes du dataset ; 

In [13]:
df.tail()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
502,1107,2025-01-27 12:00:00,B2,École,C,29.8,66.5,778.0,48.0,243.5,Normal,Normal,Lundi,Non
503,1271,2025-03-09 12:00:00,B3,Hôpital,B,21.1,66.3,3900.0,53.0,145.8,Eco,Alerte,Dimanche,Oui
504,1349,2025-03-29 00:00:00,B5,Centre commercial,B,21.6,61.9,959.0,33.0,100.2,Eco,Normal,Samedi,Non
505,1436,2025-04-19 18:00:00,B7,Bureau,C,21.2,54.5,1066.0,38.0,150.9,Boost,Normal,Samedi,Oui
506,1103,2025-01-26 12:00:00,B2,École,D,NaN,64.8,584.0,29.0,163.4,Normal,Panne,Dimanche,Non


4) Combien d'observations contient le dataset ?

In [14]:
df.shape[0]

507

5) Combien de variables possède le dataset ? 

In [15]:
df.shape[1]

14

6) Identifier les variables numériques 

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 507 entries, 0 to 506
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_mesure           507 non-null    int64  
 1   date                507 non-null    str    
 2   batiment            507 non-null    str    
 3   type_batiment       503 non-null    str    
 4   zone                507 non-null    str    
 5   temperature         495 non-null    float64
 6   humidite            496 non-null    float64
 7   co2                 500 non-null    float64
 8   occupation          501 non-null    float64
 9   consommation_kwh    502 non-null    float64
 10  mode_climatisation  502 non-null    str    
 11  etat_systeme        507 non-null    str    
 12  jour_semaine        502 non-null    str    
 13  alerte              507 non-null    str    
dtypes: float64(5), int64(1), str(8)
memory usage: 55.6 KB


les variables numériquues sont:  
[id_mesure]   
[temperature]
[humidite]   
[co2] 
[occupation]  
[consommation_kwh]

7) Identifier les variables catégorielles 


Les variables catégorielles sont :

[mode_climatisation]    
[etat_systeme]  
[jour_semaine]   
[alerte]

8) Identifier les dates 

Les variables dates sont:

[date] 

9) Identifier les identifiants

Les identifiants sont:

[id_mesure]

10) Déterminer les statistiques : moyenne, médiane, minimum, maximum, écart-type et quartiles. 

In [18]:
num = df.select_dtypes(include='number')

stats = pd.DataFrame({
    'moyenne':    num.mean(),
    'médiane':    num.median(),
    'minimum':    num.min(),
    'maximum':    num.max(),
    'écart-type': num.std(),
    'Q1 (25%)':   num.quantile(0.25),
    'Q3 (75%)':   num.quantile(0.75),
})

print(stats)

                      moyenne  médiane  minimum  maximum  écart-type  \
id_mesure         1251.114398  1252.00   1001.0   1500.0  144.782769   
temperature         24.154141    24.00    -30.0     96.0    7.418465   
humidite            57.864113    57.55    -12.0    160.0   16.026336   
co2                844.150000   787.50     89.0   6000.0  582.181386   
occupation          44.850299    46.00    -20.0    116.0   24.949139   
consommation_kwh   169.069323   169.80   -100.0    336.2   53.164294   

                  Q1 (25%)  Q3 (75%)  
id_mesure         1125.500  1376.500  
temperature         21.600    26.000  
humidite            49.275    65.750  
co2                623.750   952.000  
occupation          27.000    61.000  
consommation_kwh   136.875   202.975  


11) Y a-t-il des variables potentiellement problématiques ? 

La variable [date] peut etre considere comme problematique tenant compte du type qui es : "str" 

12) Pour les données incohérentes : 

a) rechercher des valeurs telles que humidité < 0 

In [19]:
df[df['humidite'] < 0]

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
281,1126,2025-02-01 06:00:00,B2,École,D,24.8,-5.0,759.0,57.0,225.2,Boost,Normal,Samedi,Non
335,1036,2025-01-09 18:00:00,B4,Bureau,D,21.2,-8.0,160.0,25.0,120.2,Normal,Alerte,Jeudi,Non
366,1216,2025-02-23 18:00:00,B3,Hôpital,B,23.2,-12.0,702.0,41.0,119.8,Normal,Normal,Dimanche,Non


b) rechercher des valeurs telles que humidité > 100  

In [20]:
df[df['humidite'] <100]

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
0,1174,2025-02-13 06:00:00,B8,Entrepôt,A,23.0,40.0,851.0,26.0,138.4,Eco,Normal,Jeudi,Non
1,1275,2025-03-10 12:00:00,B7,Bureau,D,28.0,69.0,538.0,0.0,170.2,Eco,Normal,Lundi,Non
2,1493,2025-05-04 00:00:00,B5,Centre commercial,B,19.5,52.3,1198.0,64.0,149.8,Normal,Normal,Dimanche,Oui
3,1073,2025-01-19 00:00:00,B6,Université,D,20.0,58.4,1014.0,30.0,141.6,Normal,Normal,Dimanche,Non
4,1454,2025-04-24 06:00:00,B7,Bureau,D,26.8,70.5,628.0,42.0,228.3,Normal,Normal,Jeudi,Non
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
502,1107,2025-01-27 12:00:00,B2,École,C,29.8,66.5,778.0,48.0,243.5,Normal,Normal,Lundi,Non
503,1271,2025-03-09 12:00:00,B3,Hôpital,B,21.1,66.3,3900.0,53.0,145.8,Eco,Alerte,Dimanche,Oui
504,1349,2025-03-29 00:00:00,B5,Centre commercial,B,21.6,61.9,959.0,33.0,100.2,Eco,Normal,Samedi,Non
505,1436,2025-04-19 18:00:00,B7,Bureau,C,21.2,54.5,1066.0,38.0,150.9,Boost,Normal,Samedi,Oui


c) rechercher des valeurs telles que température extrêmement élevée ;